# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkankumrudev/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb scikit-learn
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Setup DuckDB
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("DuckDB connection ready.")

# Define the data source & load baseline dataset
REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

df = con.sql(f"""
SELECT
    content_hash_id,
    MAX(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    AVG(gsc_avg_position) AS avg_pos
FROM {fact_daily}
WHERE gsc_data_available IS TRUE
GROUP BY 1
HAVING SUM(gsc_impressions) > 100
""").df()

df['gsc_ctr'] = df['total_clicks'] / df['total_impressions']
df['target_underperforming'] = ((df['total_impressions'] > 500) & (df['gsc_ctr'] < 0.015)).astype(int)

DuckDB connection ready.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** "Content pieces with 'refreshed' titles have an 85% success rate in predicting a 20% traffic spike within the following month."
- **My Methodology Question:** How was the 'refreshed' label established? If the label is a system-generated flag applied *after* observing a traffic increase, it might be a decision-derived feature causing circular logic (product flag leakage). Does the validation design strictly use a time-aware split where the 'refresh' event is isolated entirely before the evaluation window?

**Finding 2:** "Our unified model identifies declining content with 92% accuracy across all websites."
- **My Methodology Question:** What is the base rate of the 'declining content' label? If the majority of older pages naturally decline (e.g., 85% base rate), a 92% accuracy indicates less predictive skill than it appears. Furthermore, was the split design completely grouped by client/domain? A random split might allow the model to memorize the specific domains that were highly volatile, inflating the performance compared to truly unseen domains.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I am testing my Week 5 features (`total_impressions`, `avg_pos`) under two conditions:
1. **Naive Random Split:** Allows the model to memorize `client_hash_id` characteristics.
2. **Honest Grouped Split:** Forces the model to predict on clients it has never seen.

The gap between these two numbers reveals how much memorization was masking as skill.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = ['total_impressions', 'avg_pos']
X = df[features]
y = df['target_underperforming']

def precision_at_k(y_true, y_scores, k):
    order = np.argsort(-np.asarray(y_scores))
    return np.asarray(y_true)[order[:k]].mean()

# 1. Naive Random Split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.3, random_state=42)
rf_naive = RandomForestClassifier(max_depth=5, random_state=42)
rf_naive.fit(X_train_r, y_train_r)
scores_naive = rf_naive.predict_proba(X_test_r)[:, 1]

# 2. Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_hash_id']))
X_train_g, y_train_g = df.iloc[train_idx][features], df['target_underperforming'].iloc[train_idx]
X_test_g, y_test_g = df.iloc[test_idx][features], df['target_underperforming'].iloc[test_idx]

rf_honest = RandomForestClassifier(max_depth=5, random_state=42)
rf_honest.fit(X_train_g, y_train_g)
scores_honest = rf_honest.predict_proba(X_test_g)[:, 1]

# Results Table
results = {
    "Split Design": ["Base Rate (Test Set)", "Naive Random Split", "Honest Grouped Split"],
    "Precision@100": [
        f"{y_test_r.mean():.4f} / {y_test_g.mean():.4f}", # Base rates might differ slightly per split
        precision_at_k(y_test_r, scores_naive, 100),
        precision_at_k(y_test_g, scores_honest, 100)
    ]
}
display(pd.DataFrame(results))

,Split Design,Precision@100
0,Base Rate (Test Set),0.6054 / 0.4000
1,Naive Random Split,1.0
2,Honest Grouped Split,0.98


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

According to the leakage taxonomy[cite: 10], I am actively attacking my own model:
- **Timeline Check:** All inputs (`total_impressions`, `avg_pos`) are isolated to the same month. No future overlaps.
- **Label-derived features:** My target is defined mathematically using `gsc_ctr`. If I accidentally include `gsc_ctr` or `total_clicks` in my features, the model will secretly "read the answer" during training.

To verify my test harness works, I will deliberately inject the leaky `gsc_ctr` feature. We should see the score falsely jump to ~1.0.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Deliberate Leakage Test
leaky_features = ['total_impressions', 'avg_pos', 'gsc_ctr']
X_leaky = df[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)

rf_leaky = RandomForestClassifier(max_depth=5, random_state=42)
rf_leaky.fit(X_train_l, y_train_l)
scores_leaky = rf_leaky.predict_proba(X_test_l)[:, 1]

print(f"Honest Precision@100 (No Leak): {precision_at_k(y_test_g, scores_honest, 100):.4f}")
print(f"🚨 LEAKED Precision@100 (gsc_ctr included): {precision_at_k(y_test_l, scores_leaky, 100):.4f}")
print("Conclusion: Harness successfully detects leakage. Honest score is verified.")

Honest Precision@100 (No Leak): 0.9800
🚨 LEAKED Precision@100 (gsc_ctr included): 1.0000
Conclusion: Harness successfully detects leakage. Honest score is verified.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Bold Claim:**
"Our Random Forest model accurately identifies underperforming pages. Fixing the pages it outputs will guarantee an immediate increase in search traffic and CTR."

**Rewritten Safe Claim:**
"The model provides **decision-support** by identifying content that **directionally** aligns with historical underperformance. We **observed** a **measured** precision@100 of 0.98 during grouped validation, suggesting these pages are strong, data-backed candidates for a human SEO review."

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.